1. Print out the last five elements in the lists trainde and trainen.

In [1]:
import requests, os, tarfile

url=("https://raw.githubusercontent.com/neychev/"
     "small_DL_repo/master/datasets/Multi30k/training.tar.gz")    #The URL to download the training dataset
os.makedirs("files", exist_ok=True)
if not os.path.exists("files/training.tar.gz"):    #Downloads the dataset
    fb1=requests.get(url)
    with open("files/training.tar.gz","wb") as f:
        f.write(fb1.content)
train=tarfile.open('files/training.tar.gz')    #Unzips the file
train.extractall('files')    #Places content in the files folder
train.close()

/tmp/ipykernel_620/3577882730.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  train.extractall('files')    #Places content in the files folder


In [2]:
with open("files/train.de", 'rb') as fb:
    trainde = fb.readlines()
with open("files/train.en", 'rb') as fb:
    trainen = fb.readlines()
trainde=[i.decode("utf-8").strip() for i in trainde]
trainen=[i.decode("utf-8").strip() for i in trainen]

In [3]:
from pprint import pprint

print("The last five elements of the list trainde are:")
pprint(trainde[-5:])

print("The last five elements of the list trainen are:")
pprint(trainen[-5:])

The last five elements of the list trainde are:
['Ein Bergsteiger übt an einer Kletterwand.',
 'Zwei Bauarbeiter arbeiten auf einer Straße vor einem Hauses.',
 'Ein älterer Mann sitzt mit einem Jungen mit einem Wagen vor einer Fassade.',
 'Ein Mann in Shorts und Hawaiihemd lehnt sich über das Geländer eines '
 'Lotsenboots, mit Nebel und Bergen im Hintergrund.',
 '']
The last five elements of the list trainen are:
['A rock climber practices on a rock climbing wall.',
 "Two male construction workers are working on a street outside someone's home",
 'An elderly man sits outside a storefront accompanied by a young boy with a '
 'cart.',
 'A man in shorts and a Hawaiian shirt leans over the rail of a pilot boat, '
 'with fog and mountains in the background.',
 '']


2.   Convert the second German phrase in `trainde` into a list of tokens, and name the list `tokenized_de1`. Then, convert the second English phrase in `trainen` into a list of tokens, and name the list `tokenized_en1`. Print out the lists `tokenized_de1` and `tokenized_en1`.

In [4]:
import os, spacy

try:
    de_tokenizer = spacy.load("de_core_news_sm") #Tries to load the German model
except IOError:
    os.system("python -m spacy download de_core_news_sm") #If the German model isn’t found, downloads it
    de_tokenizer = spacy.load("de_core_news_sm")

try:
    en_tokenizer = spacy.load("en_core_web_sm") #Tries to load the English model
except IOError:
    os.system("python -m spacy download en_core_web_sm") # If the English model isn’t found, downloads it
    en_tokenizer = spacy.load("en_core_web_sm")

In [5]:
tokenized_de=[tok.text for tok in
              de_tokenizer.tokenizer(trainde[1])]
tokenized_en=[tok.text for tok in
              en_tokenizer.tokenizer(trainen[1])]
print(tokenized_de)
print(tokenized_en)

['Mehrere', 'Männer', 'mit', 'Schutzhelmen', 'bedienen', 'ein', 'Antriebsradsystem', '.']
['Several', 'men', 'in', 'hard', 'hats', 'are', 'operating', 'a', 'giant', 'pulley', 'system', '.']


3. Use the dictionary en_word_dict to convert tokenized_en1 that we created in Exercise 2 into a list of indices enidx1. Then, convert enidx1 back into a list of tokens entokens1 using the dictionary en_idx_dict. Finally, join the tokens in entokens1 into a sentence.

In [6]:
from collections import Counter

en_tokens=[["BOS"]+[tok.text for tok in en_tokenizer.tokenizer(x)]
           +["EOS"] for x in trainen] #Adds BOS and EOS at the beginning and end of each phrase, respectively
PAD=0
UNK=1
word_count=Counter()
for sentence in en_tokens:
    for word in sentence:
        word_count[word]+=1
frequency=word_count.most_common(50000)
total_en_words=len(frequency)+2
en_word_dict={w[0]:idx+2 for idx,w in enumerate(frequency)} #Assigns an index to each unique token
en_word_dict["PAD"]=PAD
en_word_dict["UNK"]=UNK #The padding token and unknown tokens are assigned indices 0 and 1, respectively
en_idx_dict={v:k for k,v in en_word_dict.items()} #A dictionary to map indices back to tokens

In [7]:
enidx=[en_word_dict.get(i,UNK) for i in tokenized_en]
print(enidx)

[164, 36, 7, 335, 286, 17, 1208, 2, 753, 3933, 2710, 5]


In [8]:
entokens=[en_idx_dict.get(i,"UNK") for i in enidx]
print(entokens)
en_phrase=" ".join(entokens)
for x in '''?:;.,'("-!&)%''':
    en_phrase=en_phrase.replace(f" {x}",f"{x}")
print(en_phrase)

['Several', 'men', 'in', 'hard', 'hats', 'are', 'operating', 'a', 'giant', 'pulley', 'system', '.']
Several men in hard hats are operating a giant pulley system.


4. Use the dictionary de_word_dict to convert tokenized_de1 that we created in Exercise 2 into a list of indices deidx1. Then, convert deidx1 back into a list of tokens detokens1 using the dictionary de_idx_dict. Finally, join the tokens in detokens1 into a sentence.

In [9]:
de_tokens=[["BOS"]+[tok.text for tok in de_tokenizer.tokenizer(x)]
           +["EOS"] for x in trainde] #Adds BOS and EOS at the beginning and end of each phrase, respectively
de_word_count=Counter()
for sentence in de_tokens:
    for word in sentence:
        de_word_count[word]+=1
defrequency=de_word_count.most_common(50000)
total_de_words=len(defrequency)+2
de_word_dict={w[0]:idx+2 for idx,w in enumerate(defrequency)} #Assigns an index to each unique token
de_word_dict["PAD"]=PAD
de_word_dict["UNK"]=UNK #The padding token and unknown tokens are assigned indices 0 and 1, respectively
de_idx_dict={v:k for k,v in de_word_dict.items()} #A dictionary to map indices back to tokens

In [10]:
deidx=[de_word_dict.get(i,UNK) for i in tokenized_de]
print(deidx)

[84, 31, 10, 838, 2096, 15, 8014, 4]


In [11]:
detokens=[de_idx_dict.get(i,"UNK") for i in deidx]
print(detokens)
de_phrase=" ".join(detokens)

for x in '''?:;.,'("-!&)%''':
    de_phrase=de_phrase.replace(f" {x}",f"{x}")
print(de_phrase)

['Mehrere', 'Männer', 'mit', 'Schutzhelmen', 'bedienen', 'ein', 'Antriebsradsystem', '.']
Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.


5. Use the de2en() function to translate the 11th and 12th sentences in trainde into English. Compare the translations with the original English translations in trainen.

In [15]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import numpy as np
def subsequent_mask(size):
    attn_shape = (1, size, size)
    subsequent_mask = np.triu(np.ones(attn_shape),
                              k=1).astype('uint8')
    output = torch.from_numpy(subsequent_mask) == 0
    return output

def make_std_mask(tgt, pad):
    tgt_mask=(tgt != pad).unsqueeze(-2)
    output=tgt_mask & subsequent_mask(\
        tgt.size(-1)).type_as(tgt_mask.data)
    return output

class Batch:
    def __init__(self, src, trg=None, pad=0):
        src = torch.from_numpy(src).to(DEVICE).long()
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2) #Creates a source mask to hide padding at the end of the sentence
        if trg is not None:
            trg = torch.from_numpy(trg).to(DEVICE).long()
            self.trg = trg[:, :-1] #Creates input to the decoder
            self.trg_y = trg[:, 1:] #Shifts the input one token to the right and uses it as output
            self.trg_mask = make_std_mask(self.trg, pad) #Creates a target mask
            self.ntokens = (self.trg_y != pad).data.sum()

In [17]:
import math
from torch import nn

class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        out = self.lut(x) * math.sqrt(self.d_model)
        return out

In [18]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000): #Initiates the class, allowing a maximum of 5,000 positions
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=DEVICE)
        position = torch.arange(0., max_len,
                                device=DEVICE).unsqueeze(1)
        div_term = torch.exp(torch.arange(
            0., d_model, 2, device=DEVICE)
            * -(math.log(10000.0) / d_model))
        pe_pos = torch.mul(position, div_term)
        pe[:, 0::2] = torch.sin(pe_pos) #Applies the sine function to even indices in the vector
        pe[:, 1::2] = torch.cos(pe_pos) #Applies the cosine function to odd indices in the vector
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)].requires_grad_(False)
        out = self.dropout(x)
        return out

In [19]:
def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query,
              key.transpose(-2, -1)) / math.sqrt(d_k) #The scaled attention score is the dot product of query and key, scaled by the square root of d_k
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9) #If there’s a mask, hide future elements in the sequence
    p_attn = nn.functional.softmax(scores, dim=-1) #Calculates attention weights
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn #Returns both attention and attention weights

In [20]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super().__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = nn.ModuleList([deepcopy(
            nn.Linear(d_model, d_model)) for i in range(4)])
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [l(x).view(nbatches, -1, self.h,
           self.d_k).transpose(1, 2)
         for l, x in zip(self.linears, (query, key, value))]
        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(
            nbatches, -1, self.h * self.d_k)
        output = self.linears[-1](x)
        return output

In [21]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h1 = self.w_1(x)
        h2 = self.dropout(h1)
        return self.w_2(h2)

In [22]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from torch import nn

class Transformer(nn.Module):
    def __init__(self, encoder, decoder,
                 src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder #Defines an encoder in the transformer
        self.decoder = decoder #Defines a decoder in the transformer
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt),
                            memory, src_mask, tgt_mask)

    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask) #The source language is encoded into an abstract vector representation by the encoder
        output = self.decode(memory, src_mask, tgt, tgt_mask) #The decoder uses the vector representation to generate the translation in the target language
        return output

In [23]:
from copy import deepcopy

class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList(
            [deepcopy(layer) for i in range(N)])
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
            output = self.norm(x)
        return output

In [24]:
class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = nn.ModuleList([deepcopy(
        SublayerConnection(size, dropout)) for i in range(2)])
        self.size = size

    def forward(self, x, mask):
        x = self.sublayer[0](
            x, lambda x: self.self_attn(x, x, x, mask))
        output = self.sublayer[1](x, self.feed_forward)
        return output

class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        output = x + self.dropout(sublayer(self.norm(x)))
        return output

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        x_zscore = (x - mean) / torch.sqrt(std ** 2 + self.eps)
        output = self.a_2*x_zscore+self.b_2
        return output

In [25]:
class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList(
            [deepcopy(layer) for i in range(N)])
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        output = self.norm(x)
        return output

In [26]:
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn,
                 feed_forward, dropout):
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = nn.ModuleList([deepcopy(
        SublayerConnection(size, dropout)) for i in range(3)])

    def forward(self, x, memory, src_mask, tgt_mask):
        x = self.sublayer[0](x, lambda x:
                 self.self_attn(x, x, x, tgt_mask)) #The first sublayer is a masked multi-head attention layer
        x = self.sublayer[1](x, lambda x:
                 self.src_attn(x, memory, memory, src_mask)) #The second sublayer is a cross-attention layer between the two languages
        output = self.sublayer[2](x, self.feed_forward) #The third sublayer is a feed-forward network
        return output

In [27]:
class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        out = self.proj(x)
        probs = nn.functional.log_softmax(out, dim=-1)
        return probs

In [36]:
def create_model(src_vocab, tgt_vocab, N, d_model,
                 d_ff, h, dropout=0.1):
    attn=MultiHeadedAttention(h, d_model).to(DEVICE)
    ff=PositionwiseFeedForward(d_model, d_ff, dropout).to(DEVICE)
    pos=PositionalEncoding(d_model, dropout).to(DEVICE)
    model = Transformer(
        Encoder(EncoderLayer(d_model,deepcopy(attn),deepcopy(ff),
                             dropout).to(DEVICE),N).to(DEVICE), #Creates an encoder by instantiating the Encoder class
        Decoder(DecoderLayer(d_model,deepcopy(attn),
             deepcopy(attn),deepcopy(ff), dropout).to(DEVICE),
                N).to(DEVICE), #Creates a decoder by instantiating the Decoder class
        nn.Sequential(Embeddings(d_model, src_vocab).to(DEVICE),
                      deepcopy(pos)), #Creates src_embed by generating input embeddings for the source language
        nn.Sequential(Embeddings(d_model, tgt_vocab).to(DEVICE),
                      deepcopy(pos)), #Creates tgt_embed by generating input embeddings for the target language
        Generator(d_model, tgt_vocab)).to(DEVICE) #Creates a generator by instantiating the Generator class
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model.to(DEVICE)

model = create_model(total_de_words, total_en_words, 2, 256, 512, 8)

In [37]:
class LabelSmoothing(nn.Module):
    def __init__(self, size, padding_idx, smoothing=0.0):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction='sum')
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size
        self.true_dist = None

    def forward(self, x, target):
        assert x.size(1) == self.size
        true_dist = x.data.clone()
        true_dist.fill_(self.smoothing / (self.size - 2))
        true_dist.scatter_(1,
               target.data.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        self.true_dist = true_dist
        output = self.criterion(x, true_dist.clone().detach())
        return output

In [38]:
class NoamOpt:
    def __init__(self, model_size, factor, warmup, optimizer):
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.model_size = model_size
        self._rate = 0

    def step(self):
        self._step += 1
        rate = self.rate()
        for p in self.optimizer.param_groups:
            p['lr'] = rate
        self._rate = rate
        self.optimizer.step()

    def rate(self, step=None):
        if step is None:
            step = self._step
        output = self.factor * (self.model_size ** (-0.5) *
        min(step ** (-0.5), step * self.warmup ** (-1.5)))
        return output

In [39]:
class SimpleLossCompute:
    def __init__(self, generator, criterion, opt=None):
        self.generator = generator
        self.criterion = criterion
        self.opt = opt

    def __call__(self, x, y, norm):
        x = self.generator(x)
        loss = self.criterion(x.contiguous().view(-1, x.size(-1)),
                              y.contiguous().view(-1)) / norm
        loss.backward()
        if self.opt is not None:
            self.opt.step()
            self.opt.optimizer.zero_grad()
        return loss.data.item() * norm.float()

In [43]:
import torch

optimizer = NoamOpt(256, 1, 2000, torch.optim.Adam(
    model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))
criterion = LabelSmoothing(total_en_words,
                           padding_idx=0, smoothing=0.0)
loss_func = SimpleLossCompute(
            model.generator, criterion, optimizer)

In [44]:
def data_gen(src, tgt, batch_size, PAD):
    num_batches = math.ceil(len(src) / batch_size)
    indices = list(range(len(src)))
    np.random.shuffle(indices)

    for i in range(num_batches):
        batch_indices = indices[i * batch_size : (i + 1) * batch_size]

        # Get sentences for the current batch
        src_batch_raw = [src[idx] for idx in batch_indices]
        tgt_batch_raw = [tgt[idx] for idx in batch_indices]

        # Pad sequences to the maximum length within the current batch
        max_src_len = max(len(s) for s in src_batch_raw)
        max_tgt_len = max(len(t) for t in tgt_batch_raw)

        src_batch_padded = [s + [PAD] * (max_src_len - len(s)) for s in src_batch_raw]
        tgt_batch_padded = [t + [PAD] * (max_tgt_len - len(t)) for t in tgt_batch_raw]

        # Convert to numpy arrays
        src_batch_np = np.array(src_batch_padded, dtype=np.int64)
        tgt_batch_np = np.array(tgt_batch_padded, dtype=np.int64)

        yield Batch(src_batch_np, tgt_batch_np, PAD)

In [45]:
# Convert tokenized sentences to indexed sentences
de_indexed_sentences = []
for sentence_tokens in de_tokens:
    indexed_sentence = [de_word_dict.get(token, UNK) for token in sentence_tokens]
    de_indexed_sentences.append(indexed_sentence)

en_indexed_sentences = []
for sentence_tokens in en_tokens:
    indexed_sentence = [en_word_dict.get(token, UNK) for token in sentence_tokens]
    en_indexed_sentences.append(indexed_sentence)

# Create batches
batch_size = 32 # You can adjust this batch size
batches = list(data_gen(de_indexed_sentences, en_indexed_sentences, batch_size, PAD))

print(f"Number of batches created: {len(batches)}")

Number of batches created: 907


In [49]:
for epoch in range(50):
    model.train()
    tloss=0
    tokens=0
    for batch in batches:
        out = model(batch.src, batch.trg,
                    batch.src_mask, batch.trg_mask) #Predicts the next token using the transformer
        loss = loss_func(out, batch.trg_y, batch.ntokens) #Calculates loss and adjusts model parameters
        tloss += loss
        tokens += batch.ntokens #Counts the number of tokens in the batch
    print(f"Epoch {epoch}, average loss: {tloss/tokens}")
torch.save(model.state_dict(),"files/de2en.pth") #Saves the weights in the trained model after training

Epoch 0, average loss: 4.442167282104492
Epoch 1, average loss: 2.812715530395508


KeyboardInterrupt: 

In [ ]:
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
def de2en(ger):
    tokenized_ger= [tok.text for tok in de_tokenizer.tokenizer(ger)]
    tokenized_ger=["BOS"]+tokenized_ger+["EOS"]
    geridx=[de_word_dict.get(i,UNK) for i in tokenized_ger]
    src=torch.tensor(geridx).long().to(DEVICE).unsqueeze(0)
    src_mask=(src!=0).unsqueeze(-2)
    memory=model.encode(src,src_mask)    #Uses the encoder to convert German to vector representations
    start_symbol=en_word_dict["BOS"]
    ys = torch.ones(1, 1).fill_(start_symbol).type_as(src.data)
    translation=[]
    for i in range(100):
        out = model.decode(memory,src_mask,ys,
        subsequent_mask(ys.size(1)).type_as(src.data))
        prob = model.generator(out[:, -1])    #Predicts the next English token using the decoder
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, torch.ones(1, 1).type_as(
            src.data).fill_(next_word)], dim=1)
        sym = en_idx_dict[ys[0, -1].item()]
        if sym != 'EOS':    #Stops translating when the next token is EOS
            translation.append(sym)
        else:
            break
    trans=" ".join(translation)
    for x in '''?:;.,'("-!&)%''':
        trans=trans.replace(f" {x}",f"{x}")    #Joins the predicted tokens to form an English sentence as the translation
    return trans

In [ ]:
# Încărcăm modelul (poți sări peste primele două rânduri dacă l-ai rulat deja mai sus în notebook)
model.load_state_dict(torch.load("files/de2en.pth",
            weights_only=True, map_location=DEVICE))
model.eval()

# Trecem prin indicii 10 și 11 (adică a 11-a și a 12-a propoziție)
for i in range(10, 12):
    print("original Ger:", trainde[i])
    print("original Eng:", trainen[i])
    print("translated Eng:", de2en(trainde[i]))
    print("-" * 50) # Am adăugat o linie despărțitoare pentru a citi mai ușor rezultatele